# CONFIRMACIÓN ENTRADA LISTA DE SÍMBOLOS PARA REVERSIONES FUTUROS - BINANCE

In [1]:
# =========================================================
# LIBRERÍAS
# =========================================================
import requests
import os
import pandas as pd
import numpy as np
import ta
import json
from pathlib import Path

import time
from datetime import datetime, timezone, timedelta

import winsound

In [2]:
# =========================================================
# CONFIGURACIÓN GLOBAL
# =========================================================
BINANCE_FUTURES_BASE = "https://fapi.binance.com"
INTERVAL = "15m"                 # coherente con scanner 1H
LIMIT = 120                     # ~15 días de contexto
LOOKBACK = 120
MAX_SYMBOLS_TRAIN = 540       # limitar universo de entrenamiento
MIN_NOTIONAL_VOLUME = 10_000  # Sugerido 5_000_000

AVG_LIMIT_LONG = 30    # límite de KPI AVG para reversión LONG 
AVG_LIMIT_SHORT = 70    # límite de KPI AVG para reversión SHORT 

FILEPATH = "C:/Users/Lenovo S540/Python_scripts/ListaSimbolosReversionesFuturos.txt"

In [3]:
# =========================================================
# MANEJO DE LISTA DE SÍMBOLOS
# =========================================================
def load_symbol_map(filepath: str) -> dict:
    """
    Carga archivo JSON con formato:
    {
        "ETHUSDT": "LONG",
        "LTCUSDT": "SHORT"
    }
    """
    try:
        path = Path(filepath)

        if not path.exists():
            return {}

        with open(path, "r") as f:
            data = json.load(f)

        # Validación básica
        valid = {}
        for k, v in data.items():
            if v in ["LONG", "SHORT"]:
                valid[k] = v

        return valid

    except Exception as e:
        print(f"[WARN] Error cargando JSON: {e}")
        return {}


def save_symbol_map(filepath: str, symbol_map: dict):
    """
    Guarda el diccionario actualizado en formato JSON.
    """
    try:
        with open(filepath, "w") as f:
            json.dump(symbol_map, f, indent=4)
    except Exception as e:
        print(f"[WARN] Error guardando JSON: {e}")


In [4]:
# =========================================================
# ENVÍO DE ALERTAS
# =========================================================
TELEGRAM_BOT_TOKEN = "8593522900:AAFSgAMZAKolZaYqm2GjeqY4Hr3tP7Jnk2M"
TELEGRAM_CHAT_ID = "1733579121"

def send_telegram_alert(message: str):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "Markdown"
    }
    try:
        requests.post(url, json=payload, timeout=5)
    except Exception as e:
        print(f"[WARN] Telegram alert failed: {e}")



def local_warning():

    frequency = 2500  # Set Frequency To 2500 Hertz
    duration = 1000   # Set Duration To 1000 ms == 1 second

    winsound.Beep(frequency, duration)
    print("Oportunidades de entrada encontradas!")
    
    # Example of beeping with time delays
    for i in range(3):
        winsound.Beep(500, 200) # Frequency 500 Hz, duration 200ms
        time.sleep(0.5)        # Pause for 0.5 seconds between beeps


In [5]:
# =========================================================
# 2️⃣ OHLC
# =========================================================
def get_ohlc(
    symbol: str,
    interval: str = "1h",
    limit: int = 500,
    max_total: int | None = None
):
    """
    Descarga OHLCV desde Binance Futures (USDT-M).
    
    - Usa requests (sin API key)
    - Soporta paginación si max_total > limit
    - Devuelve DataFrame limpio y ordenado
    """

    url = f"{BINANCE_FUTURES_BASE}/fapi/v1/klines"
    all_data = []
    end_time = None

    max_total = max_total or limit

    while len(all_data) < max_total:
        fetch = min(1500, max_total - len(all_data))

        params = {
            "symbol": symbol,
            "interval": interval,
            "limit": fetch
        }

        if end_time is not None:
            params["endTime"] = end_time

        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()
            
            if not data:
                break

            all_data = data + all_data
            end_time = data[0][0] - 1  # ms, vela anterior

        except Exception:
            break

    if len(all_data) < 50:
        return None

    df = pd.DataFrame(
        all_data,
        columns=[
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "close_time",
            "quote_asset_volume",
            "num_trades",
            "taker_buy_base_volume",
            "taker_buy_quote_volume",
            "ignore"
        ]
    )

    # ---------- Tipos ----------
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)

    # ---------- Timestamps ----------
    df["time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)

    df = df[["time", "open", "high", "low", "close", "volume"]]
    df.sort_values("time", inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.dropna(inplace=True)

    if len(df) < 50:
        return None

    return df


In [6]:
def df15_provider(symbol: str, limit: int = 120):
    """
    Provee dataframe OHLCV de 15m con indicadores técnicos
    necesarios para confirmación y monitoreo.

    Retorna:
        pd.DataFrame o None
    """

    df = get_ohlc(symbol, interval="15m", limit=limit)

    if df is None or len(df) < 50:
        return None

    df = df.sort_index()

    # eliminar vela viva
    now_utc = datetime.now(timezone.utc)
    if df.iloc[-1]["time"] + pd.Timedelta(minutes=15) > now_utc:
        df = df.iloc[:-1]

    # ---------------- INDICADORES ---------------- #
    df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()

    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / (avg_loss + 1e-9)
    df["rsi"] = 100 - (100 / (1 + rs))

    ema_fast = df["close"].ewm(span=12, adjust=False).mean()
    ema_slow = df["close"].ewm(span=26, adjust=False).mean()

    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=9, adjust=False).mean()

    df["macd_hist"] = macd - macd_signal

    return df


In [7]:
def compute_atr_from_df(df: pd.DataFrame, period: int = 14):
    if len(df) < period + 1:
        return None

    df = df.copy()
    df["prev_close"] = df["close"].shift(1)

    tr = np.maximum.reduce([
        df["high"] - df["low"],
        (df["high"] - df["prev_close"]).abs(),
        (df["low"] - df["prev_close"]).abs()
    ])

    atr = pd.Series(tr).rolling(period).mean().iloc[-1]

    if pd.isna(atr) or atr <= 0:
        return None

    return float(atr)


In [8]:
def compute_macd_histogram(
    df: pd.DataFrame,
    fast: int = 12,
    slow: int = 26,
    signal: int = 9,
    price_col: str = "close"
) -> pd.Series:
    """
    Calcula histograma MACD clásico (MACD - Signal).

    Diseñado para:
    - Timing fino de entrada (5m)
    - Evaluar energía + aceleración
    - Sin repainting (solo usa datos cerrados)

    Retorna:
        pd.Series alineada con df.index
    """

    if df is None or len(df) < slow + signal + 5:
        return pd.Series(dtype="float64")

    price = df[price_col].astype(float)

    # =========================
    # EMAs
    # =========================
    ema_fast = price.ewm(span=fast, adjust=False).mean()
    ema_slow = price.ewm(span=slow, adjust=False).mean()

    # =========================
    # MACD
    # =========================
    macd = ema_fast - ema_slow
    signal_line = macd.ewm(span=signal, adjust=False).mean()

    # =========================
    # HISTOGRAMA
    # =========================
    macd_hist = macd - signal_line

    return macd_hist


In [9]:
def df5_provider(symbol: str, limit: int = 120):
    """
    Descarga velas de 5 minutos desde Binance (API pública),
    retorna DataFrame limpio y ordenado.

    Columnas:
        time (UTC datetime)
        open
        high
        low
        close
        volume

    Retorna:
        pd.DataFrame | None
    """

    try:
        df = get_ohlc(symbol, interval="5m", limit=limit)

        if df is None or len(df) < 50:
            return None


        # Eliminar vela abierta (NO cerrada)
        #now_utc = pd.Timestamp.utcnow().tz_localize("UTC")
        now_utc = datetime.now(timezone.utc)
        if df.iloc[-1]["time"] + pd.Timedelta(minutes=5) > now_utc:
            df = df.iloc[:-1]
        
        return df

    except Exception as e:
        print(f"[WARN] df5_provider {symbol}: {e}")
        return None


In [10]:
def check_follow_through_5m(df5, direction: str):
    """
    Confirma continuación mínima en 5m.
    NO redefine reversión, solo evita entrar en pullback inmediato.
    """

    if df5 is None or len(df5) < 3:
        return False

    last = df5.iloc[-1]
    prev = df5.iloc[-2]

    if direction == "LONG":
        #print(f"{last.time}: last.close ({last.close}) > prev.open ({prev.open}), prev.close ({prev.close}), last.open ({last.open})")
        return (
            last.close > max(prev.open, prev.close) and
            last.close > last.open
        )
    else:    #SHORT
        #print(f"{last.time}: last.close ({last.close}) < prev.open ({prev.open}), prev.close ({prev.close}), last.open ({last.open})")
        return (
            last.close < min(prev.open, prev.close) and
            last.close < last.open
        )


In [11]:
# def confirm_entry_15m_realtime_multi(
#     symbols_file: str,
#     df15_provider = df15_provider,
#     df5_provider = df5_provider,
#     mode: str = "early",
#     max_wait_minutes: int = 30
# ):
#     """
#     Monitorea múltiples símbolos desde archivo JSON.
#     Formato:
#         { "ETHUSDT": "LONG", "LTCUSDT": "SHORT" }

#     Cuando confirma entrada:
#         - elimina símbolo del JSON
#         - guarda archivo actualizado
#     """

#     assert mode in ["early", "conservative"]

#     start_time = datetime.now(timezone.utc)
#     max_wait = timedelta(minutes=max_wait_minutes)

#     # Parámetros por modo
#     if mode == "early":
#         min_impulse_atr = 0.6
#         max_retrace_atr = 0.35
#         min_range_rel = 0.002
#     else:
#         min_impulse_atr = 0.9
#         max_retrace_atr = 0.25
#         min_range_rel = 0.0035

#     require_follow_through = True
#     max_next_wick_atr = 0.5

#     msg =  f"🟡 MONITOREO MULTI-SÍMBOLO ({datetime.now()})\nModo: {mode}"
#     send_telegram_alert(msg)
#     print(msg)

#     # =============================
#     # LOOP PRINCIPAL
#     # =============================
#     while datetime.now(timezone.utc) - start_time < max_wait:

#         symbol_map = load_symbol_map(symbols_file)

#         if not symbol_map:
#             print("JSON vacío. Finalizando monitoreo.")
#             break

#         # Iterar copia para poder modificar dict original
#         for symbol, direction in list(symbol_map.items()):
#             # print(f"\nAnalizando {symbol}|{direction}")
#             try:
#                 # =============================
#                 # 15m DATA
#                 # =============================
#                 df15 = df15_provider(symbol, limit=120)
#                 if df15 is None or len(df15) < 50:
#                     continue

#                 last = df15.iloc[-1]
#                 prev = df15.iloc[-2]

#                 atr_15m = compute_atr_from_df(df15, period=14)
#                 if atr_15m is None or atr_15m <= 0:
#                     continue

#                 candle_range = last.high - last.low
#                 range_rel = candle_range / last.close
#                 if range_rel < min_range_rel:
#                     print(f"\n{symbol}|range_rel: {range_rel} < {min_range_rel}")
#                     continue

#                 # =============================
#                 # Impulse / Retrace 15m
#                 # =============================
#                 if direction == "LONG":
#                     impulse = (last.close - last.low) / atr_15m
#                     retrace = (last.high - last.close) / atr_15m
#                     valid_structure = last.close > last.open
#                 else:
#                     impulse = (last.high - last.close) / atr_15m
#                     retrace = (last.close - last.low) / atr_15m
#                     valid_structure = last.close < last.open

#                 if impulse < min_impulse_atr:
#                     print(f"\n{symbol}|impulse: {impulse} < {min_impulse_atr}")
#                     continue

#                 if retrace > max_retrace_atr:
#                     print(f"{symbol}|retrace: {retrace} > {max_retrace_atr}")
#                     continue

#                 if not valid_structure:
#                     print(f"{symbol}|valid_structure: {valid_structure}")
#                     continue

#                 # =============================
#                 # Barrida de liquidez 15m
#                 # =============================
#                 if direction == "LONG":
#                     adverse_wick = (last.low - prev.close) / atr_15m
#                 else:
#                     adverse_wick = (prev.close - last.high) / atr_15m

#                 if adverse_wick < -max_next_wick_atr:
#                     print(f"{symbol}|adverse_wick: {adverse_wick} < {-max_next_wick_atr}")
#                     continue

#                 # =============================
#                 # Follow Through 5m
#                 # =============================
#                 if require_follow_through:
#                     df5 = df5_provider(symbol, limit=120)
#                     if df5 is None or len(df5) < 3:
#                         continue

#                     if not check_follow_through_5m(df5, direction):
#                         print(f"{symbol}|check_follow_through_5m: FALSE")
#                         continue

#                 # =============================
#                 # MACD ENERGY 5m
#                 # =============================
#                 df5 = df5_provider(symbol, limit=150)
#                 if df5 is None or len(df5) < 120:
#                     continue

#                 macd_hist = compute_macd_histogram(df5)

#                 if macd_hist is None or len(macd_hist) < 100:
#                     continue

#                 h0 = macd_hist.iloc[-2]
#                 h1 = macd_hist.iloc[-3]
#                 h2 = macd_hist.iloc[-4]

#                 abs_hist = abs(macd_hist.iloc[-100:])

#                 # 1️⃣ Dirección correcta
#                 if direction == "LONG":
#                     direction_ok = h0 > 0
#                 else:
#                     direction_ok = h0 < 0
                
#                 if not direction_ok:
#                     print(f"{symbol}|direction_ok: {direction_ok}")
#                     continue

#                 # 2️⃣ Aceleración obligatoria
#                 accelerating = abs(h0) > abs(h1) > abs(h2)

#                 if not accelerating:
#                     print(f"{symbol}|accelerating: {accelerating}")
#                     continue

#                 # 3️⃣ Percentil dinámico adaptativo
#                 # Si está acelerando fuerte → percentil más permisivo
#                 # momentum_strength = abs(h0 - h1)
                
#                 recent_volatility = abs_hist.std()
                
#                 if recent_volatility == 0:
#                     print(f"{symbol}|recent_volatility: {recent_volatility}")
#                     continue
                
#                 acceleration_ratio = (abs(h0) - abs(h1)) / abs(h1)
                
#                 # Ajuste dinámico del percentil
#                 if acceleration_ratio > 0.8:
#                     base_percentile = 70
#                 elif acceleration_ratio > 0.4:
#                     base_percentile = 75
#                 elif acceleration_ratio >= 0.15:
#                     base_percentile = 80
#                 else:
#                     continue
                
#                 threshold = np.percentile(abs_hist, base_percentile)
                
#                 macd_energy = abs(h0)
#                 print(f"{symbol}|macd_energy: {macd_energy} > {threshold} (acceleration_ratio:{acceleration_ratio})")
#                 if macd_energy < threshold:
#                     continue

#                 # 4️⃣ Protección contra sobreextensión
#                 upper_extreme = np.percentile(abs_hist, 88)
#                 print(f"{symbol}|macd_energy: {macd_energy} < {upper_extreme}")
#                 if macd_energy > upper_extreme:
#                     # Movimiento demasiado extendido → probable agotamiento
#                     continue

#                 # =============================
#                 # CONFIRMACIÓN FINAL
#                 # =============================
#                 msg = (
#                     f"🟢 ENTRADA CONFIRMADA\n"
#                     f"{symbol} {direction} | {datetime.now()}\n"
#                     f"Modo: {mode}\n"
#                     f"Precio: {round(last.close, 6)}\n"
#                     f"ATR15m: {round(atr_15m, 6)}\n"
#                     f"Impulse: {round(impulse, 2)} ATR\n"
#                     f"Retrace: {round(retrace, 2)} ATR"
#                 )

#                 send_telegram_alert(msg)
#                 print(msg)
#                 local_warning()

#                 # Eliminar símbolo confirmado
#                 del symbol_map[symbol]
#                 save_symbol_map(symbols_file, symbol_map)

#             except Exception as e:
#                 print(f"[WARN] {symbol}: {e}")
#                 continue

#     send_telegram_alert("⏰ Finalizó monitoreo de entrada multi-símbolo (JSON).")
#     print("Monitoreo de entrada finalizado.")


In [12]:
def confirm_entry_15m_realtime_multi(
    symbols_file: str,
    df15_provider=df15_provider,
    df5_provider=df5_provider,
    mode: str = "early",
    max_wait_minutes: int = 30
):

    """
    Monitorea múltiples símbolos desde archivo JSON.

    JSON formato:
        { "ETHUSDT": "LONG", "LTCUSDT": "SHORT" }

    Cuando confirma entrada:
        - elimina símbolo del JSON
        - guarda archivo actualizado
    """

    assert mode in ["early", "conservative"]

    start_time = datetime.now(timezone.utc)
    max_wait = timedelta(minutes=max_wait_minutes)

    # =============================
    # Parámetros por modo
    # =============================
    if mode == "early":
        min_impulse_atr = 0.6
        max_retrace_atr = 0.35
        min_range_rel = 0.002
    else:
        min_impulse_atr = 0.9
        max_retrace_atr = 0.25
        min_range_rel = 0.0035

    require_follow_through = True
    max_next_wick_atr = 0.5

    msg = f"🟡 MONITOREO MULTI-SÍMBOLO ({datetime.now()})\nModo: {mode}"
    send_telegram_alert(msg)
    print(msg)

    # =============================
    # LOOP PRINCIPAL
    # =============================
    while datetime.now(timezone.utc) - start_time < max_wait:

        symbol_map = load_symbol_map(symbols_file)

        if not symbol_map:
            print("JSON vacío. Finalizando monitoreo.")
            break

        for symbol, direction in list(symbol_map.items()):

            try:

                # =============================
                # 15m DATA
                # =============================
                df15 = df15_provider(symbol, limit=120)
                if df15 is None or len(df15) < 50:
                    continue

                last = df15.iloc[-1]      # vela viva
                prev = df15.iloc[-2]

                atr_15m = compute_atr_from_df(df15, period=14)
                if atr_15m is None or atr_15m <= 0:
                    continue

                candle_range = last.high - last.low
                range_rel = candle_range / last.close

                print(f"\n{symbol}|range_rel: {range_rel} > {min_range_rel}")
                if range_rel < min_range_rel:
                    continue

                # =============================
                # IMPULSE / RETRACE
                # =============================
                if direction == "LONG":
                    impulse = (last.close - last.low) / atr_15m
                    retrace = (last.high - last.close) / atr_15m
                    valid_structure = last.close > last.open
                else:
                    impulse = (last.high - last.close) / atr_15m
                    retrace = (last.close - last.low) / atr_15m
                    valid_structure = last.close < last.open

                print(f"{symbol}|impulse: {impulse} > {min_impulse_atr}")
                if impulse < min_impulse_atr:
                    continue

                print(f"{symbol}|retrace: {retrace} < {max_retrace_atr}")
                if retrace > max_retrace_atr:
                    continue

                print(f"{symbol}|valid_structure: {valid_structure}")
                if not valid_structure:
                    continue

                # =============================
                # LIQUIDITY SWEEP PROTECTION
                # =============================
                if direction == "LONG":
                    adverse_wick = (last.low - prev.close) / atr_15m
                else:
                    adverse_wick = (prev.close - last.high) / atr_15m

                print(f"{symbol}|adverse_wick: {adverse_wick} > -{max_next_wick_atr}")
                if adverse_wick < -max_next_wick_atr:
                    continue

                # =============================
                # FOLLOW THROUGH 5m
                # =============================
                if require_follow_through:

                    df5 = df5_provider(symbol, limit=120)

                    if df5 is None or len(df5) < 5:
                        continue

                    if not check_follow_through_5m(df5, direction):
                        print(f"{symbol}|check_follow_through_5m: FALSE")
                        continue

                # =============================
                # MACD ENERGY FILTER (ROBUST)
                # =============================
                df5 = df5_provider(symbol, limit=150)

                if df5 is None or len(df5) < 120:
                    continue

                macd_hist = compute_macd_histogram(df5)

                if macd_hist is None or len(macd_hist) < 100:
                    continue

                # Usar velas CERRADAS
                h0 = macd_hist.iloc[-2]
                h1 = macd_hist.iloc[-3]
                h2 = macd_hist.iloc[-4]

                abs_hist = abs(macd_hist.iloc[-100:])

                # =============================
                # 1️⃣ Dirección
                # =============================
                if direction == "LONG":
                    direction_ok = h0 > 0
                else:
                    direction_ok = h0 < 0

                if not direction_ok:
                    print(f"{symbol}|direction_ok: FALSE")
                    continue

                # =============================
                # 2️⃣ Aceleración
                # =============================
                accelerating = abs(h0) > abs(h1) > abs(h2)
                print(f"{symbol}|accelerating: {accelerating}")
                if not accelerating:
                    continue

                # =============================
                # 3️⃣ Convexidad
                # =============================
                delta1 = abs(h0) - abs(h1)
                delta2 = abs(h1) - abs(h2)

                print(f"{symbol}|deltas: {delta1} > {delta2}")
                if delta1 <= delta2:
                    continue

                # =============================
                # 4️⃣ Percentil adaptativo
                # =============================
                acceleration_ratio = (abs(h0) - abs(h1)) / (abs(h1) + 1e-9)

                if acceleration_ratio > 1.0:
                    base_percentile = 78
                elif acceleration_ratio > 0.7:
                    base_percentile = 82
                elif acceleration_ratio > 0.4:
                    base_percentile = 85
                elif acceleration_ratio >= 0.25:
                    base_percentile = 88
                else:
                    continue

                threshold = np.percentile(abs_hist, base_percentile)

                macd_energy = abs(h0)

                print(f"{symbol}|deltas: {macd_energy} >= {threshold} (acceleration_ratio: {acceleration_ratio})")
                if macd_energy < threshold:
                    continue

                # =============================
                # 5️⃣ Zona de cola real
                # =============================
                upper_extreme = np.percentile(abs_hist, 92)

                tail_ratio = macd_energy / upper_extreme

                print(f"{symbol}|tail_ratio: {tail_ratio} >= 0.92")
                if tail_ratio < 0.92:
                    continue

                # =============================
                # 6️⃣ Z-score estructural
                # =============================
                energy_zscore = (
                    macd_energy - abs_hist.mean()
                ) / (abs_hist.std() + 1e-9)

                print(f"{symbol}|energy_zscore: {energy_zscore} >= 1.6")
                if energy_zscore < 1.6:
                    continue

                # =============================
                # CONFIRMACIÓN FINAL
                # =============================
                msg = (
                    f"🟢 ENTRADA CONFIRMADA\n"
                    f"{symbol} {direction} | {datetime.now()}\n"
                    f"Modo: {mode}\n"
                    f"Precio: {round(last.close,6)}\n"
                    f"ATR15m: {round(atr_15m,6)}\n"
                    f"Impulse: {round(impulse,2)} ATR\n"
                    f"Retrace: {round(retrace,2)} ATR"
                )

                send_telegram_alert(msg)
                print(msg)
                local_warning()

                del symbol_map[symbol]
                save_symbol_map(symbols_file, symbol_map)

            except Exception as e:
                print(f"[WARN] {symbol}: {e}")
                continue

    send_telegram_alert("⏰ Finalizó monitoreo de entrada multi-símbolo (JSON).")
    print("Monitoreo finalizado.")

In [ ]:
confirm_entry_15m_realtime_multi(FILEPATH)

🟡 MONITOREO MULTI-SÍMBOLO (2026-03-05 18:51:56.609312)
Modo: early

CHILLGUYUSDT|range_rel: 0.005797101449275294 > 0.002
CHILLGUYUSDT|impulse: 0.5999999999999943 > 0.6

HOMEUSDT|range_rel: 0.012380736029077726 > 0.002
HOMEUSDT|impulse: 1.1232091690544492 > 0.6
HOMEUSDT|retrace: 0.7507163323782174 < 0.35

LABUSDT|range_rel: 0.021881216254617787 > 0.002
LABUSDT|impulse: 0.19210399614828652 > 0.6

RIFUSDT|range_rel: 0.003440366972477123 > 0.002
RIFUSDT|impulse: 0.6536964980544869 > 0.6
RIFUSDT|retrace: 0.0 < 0.35
RIFUSDT|valid_structure: True
RIFUSDT|adverse_wick: -0.4357976653696705 > -0.5
RIFUSDT|accelerating: False

COOKIEUSDT|range_rel: 0.005437469105289296 > 0.002
COOKIEUSDT|impulse: 0.19626168224300505 > 0.6

CHILLGUYUSDT|range_rel: 0.005797101449275294 > 0.002
CHILLGUYUSDT|impulse: 0.5999999999999943 > 0.6

HOMEUSDT|range_rel: 0.012380736029077726 > 0.002
HOMEUSDT|impulse: 1.1232091690544492 > 0.6
HOMEUSDT|retrace: 0.7507163323782174 < 0.35

LABUSDT|range_rel: 0.021881216254617787 

In [ ]:
# df15 = df15_provider("REDUSDT", limit=500)
#df15
# df15.to_csv('df15_REDUSDT.csv')